In [1]:
# Import any needed libraies here!

from collections.abc import Iterable, Sequence
import time
import numpy as np
import sounddevice as sd
import json

In [2]:
# Set the default settings for your Oscilliscope/Data

sample_rate = 96000
amplitude = 0.75
json_cordinate_size = 256.0 
frame_rate = 60
sample_per_frame = sample_rate // frame_rate


In [3]:
def to_voltage(xy):
    xy = np.asarray(xy, dtype=np.float64).copy()
    xy[:, 1] = json_cordinate_size - xy[:, 1]
    return (xy / json_cordinate_size * 2.0 - 1.0) * amplitude



In [4]:


def resample_drawing(xy, n):
    np.asarray(xy, dtype=np.float64)
    src = np.linspace(0, 1, len(xy))
    dst = np.linspace(0, 1, n)

    return np.column_stack([np.interp(dst, src, xy[:, 0]),
                            np.interp(dst, src, xy[:, 1])])

In [5]:
def drawing_frames(drawing, n=sample_per_frame):
    all_points = np.vstack([np.column_stack([xs, ys]) for xs, ys in drawing])
    return to_voltage(resample_drawing(all_points, n))


In [6]:
def play_frame(frame, seconds=3):
    repeat = int(seconds * frame_rate)
    signal = np.tile(frame, (repeat, 1)).astype(np.float32)
    sd.play(signal, sample_rate)
    sd.wait()

    

In [7]:
square = [[32, 32], [224, 32], [224, 224], [32, 224], [32, 32]] 
maybe_a_circle_maybe_a_diamond = [[0, 96], [16, 112], [32, 128], [48, 144], [64, 160], [80, 144], [96, 128], [112, 112], [128, 96], [112, 80], [96, 64], [80, 48], [64, 32], [48, 48], [32, 64], [16, 80], [0, 96]]

frame = to_voltage(resample_drawing(np.asarray(maybe_a_circle_maybe_a_diamond, dtype=np.float64), sample_per_frame))
play_frame(frame)



In [12]:
with open("../../data/QuickDrawData/full_simplified_dragon.ndjson") as f:
    first = json.loads(f.readline())
frame = drawing_frames(first["drawing"])
play_frame(frame)